# Step 5: Evaluation & Visualization

Once our HeteroGNN is trained, we need to mathematically and visually verify that it learned meaningful biological relationships (e.g., *Which drugs treat which diseases?*).

This notebook covers:
1. Loading a PyTorch Lightning Checkpoint
2. Generating multi-dimensional node embeddings
3. Running evaluation metrics (ROC-AUC, Precision-Recall)
4. Dimensionality reduction via UMAP to visually analyze the Graph's Latent Space

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap.umap_ as umap

from models.hetero_gnn import BioBridgeLinkPredictor
from data.biobridge_gnn_datamodule import BioBridgeGNNDataModule

In [ ]:
# 1. Initialize datamodule to get metadata and graph structure
datamodule = BioBridgeGNNDataModule(data_dir="../data/processed/")
datamodule.setup()
metadata = datamodule.data.metadata()

# 2. Load model from Checkpoint 
# (Replace with your actual checkpoint path from the logs directory)
# ckpt_path = "../checkpoints/epoch=14-val_auroc=0.920.ckpt"
# 
# For this notebook, we initialize an untrained model to demonstrate the pipeline
print("Loading model...")
model = BioBridgeLinkPredictor(metadata=metadata, hidden_channels=128)
model.eval()
print("Model loaded successfully!")

In [ ]:
# 3. Extract Node Embeddings
# We pass all the nodes and edges through the encoder to get their latent representations.
# In a real massive graph scenario, we would use NeighborLoader for this inference step too.
with torch.no_grad():
    z_dict = model.encoder(datamodule.data.x_dict, datamodule.data.edge_index_dict)

print("Extracted Embeddings:")
for node_type, embedding in z_dict.items():
    print(f"{node_type}: {embedding.shape}")

In [ ]:
# 4. Apply UMAP to reduce 128 dimensions to 2 dimensions for plotting
reducer = umap.UMAP(random_state=42, n_neighbors=15, min_dist=0.1)

drugs = z_dict['drug'].numpy()[:500] # Take a sample so it plots fast
diseases = z_dict['disease'].numpy()[:500]

print("Fitting UMAP...")
drugs_2d = reducer.fit_transform(drugs)
diseases_2d = reducer.fit_transform(diseases)

# Prepare for Seaborn
df_drugs = pd.DataFrame(drugs_2d, columns=['UMAP 1', 'UMAP 2'])
df_drugs['Type'] = 'Drug'

df_diseases = pd.DataFrame(diseases_2d, columns=['UMAP 1', 'UMAP 2'])
df_diseases['Type'] = 'Disease'

df_viz = pd.concat([df_drugs, df_diseases])

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_viz, x='UMAP 1', y='UMAP 2', hue='Type', alpha=0.7)
plt.title('UMAP Projection of BioBridge Knowledge Graph Embeddings')
plt.show()

# If the model trained well, you will see clusters separating clearly, 
# and specific drugs clustering near the diseases they interact with!